# LOO summary table (DEG 200)

Builds a NeurIPS-ready summary table from `results/loo_summary_deg_200.csv`.

- Aggregates across seeds and held-out cell types (mean ± std).
- Bolds the best performer per metric (direction-aware).
- Renames `baseline` to `mean shift`.
- Annotates columns with ↑ (higher is better) or ↓ (lower is better).

In [1]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
DATASET_NAME = "crc"
N_DEG = 200
CSV_PATH = f"../results/loo_summary_{DATASET_NAME}_DEG_{N_DEG}.csv"
OUT_DIR = Path("../results")
OUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(CSV_PATH, engine="python")
df.head()

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,mixing_index,edistance_global,edistance_local,rmse
0,crc_120,baseline,Endothelial_CRC,200,0.503041,0.383371,0.170,0.970588,0.165,0.547060,23.927378,25.730283,13188.815073
1,crc_120,baseline,Epithelial_CRC,200,0.129327,0.227521,0.415,0.578313,0.240,0.895650,22.045196,17.380554,546598.415842
2,crc_120,baseline,Fibroblast_CRC,200,0.120677,0.202228,0.135,0.888889,0.120,0.605086,19.173983,21.201362,109470.811764
3,crc_120,baseline,Myeloid_CRC,200,0.246870,0.323619,0.165,0.818182,0.135,0.502754,22.510635,24.561290,29706.071274
4,crc_120,baseline,T_cell_CRC,200,0.676416,0.516258,0.255,0.862745,0.220,0.797638,29.482318,31.280952,14626.556810


In [5]:
# Rename holdout_celltype by replacing the last '_' in with '-'
if DATASET_NAME == "merfish":
    df["holdout_celltype"] = df["holdout_celltype"].str.replace("Fiber_tracts", "Fiber-tracts", regex=False)
else:
    df["holdout_celltype"] = df["holdout_celltype"].str.replace("T_cell", "T-cell", regex=False)

In [6]:
# Split holdout_celltype on '_' and place everything in [0] as holdout_celltype, and everything in [1:] as perturbation
df["perturbation"] = df["holdout_celltype"].apply(lambda x: "".join(x.split("_")[-1]) if len(x.split("_")) > 1 else "")
df["holdout_celltype"] = df["holdout_celltype"].apply(lambda x: "-".join(x.split("_")[:-1]))
df

,sid,model_name,holdout_celltype,n_deg,spearman,pearson,precision,direction_match,direction_match_k,mixing_index,edistance_global,edistance_local,rmse,perturbation
0,crc_120,baseline,Endothelial,200,0.503041,0.383371,0.170,0.970588,0.165,0.547060,23.927378,25.730283,13188.815073,CRC
1,crc_120,baseline,Epithelial,200,0.129327,0.227521,0.415,0.578313,0.240,0.895650,22.045196,17.380554,546598.415842,CRC
2,crc_120,baseline,Fibroblast,200,0.120677,0.202228,0.135,0.888889,0.120,0.605086,19.173983,21.201362,109470.811764,CRC
3,crc_120,baseline,Myeloid,200,0.246870,0.323619,0.165,0.818182,0.135,0.502754,22.510635,24.561290,29706.071274,CRC
4,crc_120,baseline,T-cell,200,0.676416,0.516258,0.255,0.862745,0.220,0.797638,29.482318,31.280952,14626.556810,CRC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,crc_120,spatialprop,Endothelial,200,0.478182,0.506721,0.215,0.930233,0.200,0.812261,11.400761,16.312218,31552.996000,CRC
226,crc_120,spatialprop,Epithelial,200,0.388226,0.386692,0.055,0.909091,0.050,0.034017,12.949305,15.368814,382983.620000,CRC
227,crc_120,spatialprop,Fibroblast,200,0.155347,0.264726,0.175,0.771429,0.135,0.693359,7.167927,13.263748,153635.450000,CRC
228,crc_120,spatialprop,Myeloid,200,0.504745,0.555167,0.150,0.966667,0.145,0.744805,13.912439,19.345483,77155.220000,CRC


In [7]:
# Metrics we want in the table and whether higher (+1) or lower (-1) is better.
METRICS = {
    "spearman": +1,
    "pearson": +1,
    "precision": +1,
    "direction_match_k": +1,
    "edistance_local": -1,
    #"rmse": -1,
}

PRETTY_METRIC = {
    "spearman": r"Spearman $\uparrow$",
    "pearson": r"Pearson $\uparrow$",
    "precision": r"Precision $\uparrow$",
    "direction_match_k": r"Direction Match (K) $\uparrow$",
    "edistance_local": r"E-dist (local) $\downarrow$",
    #"rmse": r"RMSE $\downarrow$",
}

# Model display order + renaming (baseline -> mean shift).
MODEL_RENAME = {
    "baseline": "mean shift",
    "cpa": "CPA",
    "scgen": "scGen",
    "spatialprop": "SpatialProp",
    "mintflow": "MintFlow",
    "cellina-ablated": "cellina (ablated)",
    "cellina-graph": "cellina (graph)",
    "cellina": "cellina",
}
MODEL_ORDER = [
    "mean shift",
    "CPA",
    "scGen",
    "SpatialProp",
    "MintFlow",
    "cellina (ablated)",
    "cellina (graph)",
    "cellina",
]

df["model_name"] = df["model_name"].map(lambda x: MODEL_RENAME.get(x, x))
df["model_name"].unique()

array(['mean shift', 'cellina (ablated)', 'cellina', 'cellina (graph)',
       'CPA', 'scGen', 'MintFlow', 'SpatialProp'], dtype=object)

In [8]:
# Two-step aggregation:
#   1) within each slide, average over held-out cell types
#   2) across slides, compute mean ± std
agg = (
    df.groupby(["perturbation", "model_name"])[list(METRICS)]
    .agg(["mean", "std"])
)

# Ensure a consistent model order within each perturbation. Build a
# MultiIndex with (perturbation, model) in the desired display order and
# reindex the aggregated table to that index.
perturbations = list(df["perturbation"].dropna().unique())
# keep stable ordering; treat empty string as its own group if present
if any(p == "" for p in perturbations):
    perturbations = sorted(perturbations, key=lambda x: (x != "", x))
else:
    perturbations = sorted(perturbations)

desired_index = pd.MultiIndex.from_product(
    [perturbations, MODEL_ORDER], names=["perturbation", "model_name"]
)
agg = agg.reindex(desired_index)

In [9]:
agg

spearman             pearson            \
                                    mean       std      mean       std   
perturbation model_name                                                  
CRC          mean shift         0.450342  0.267680  0.435388  0.270008   
             CPA                0.543982  0.195817  0.629516  0.217631   
             scGen              0.496681  0.259228  0.505983  0.363071   
             SpatialProp        0.415127  0.253309  0.472966  0.284562   
             MintFlow           0.582238  0.269537  0.659658  0.283148   
             cellina (ablated)  0.646977  0.211534  0.740287  0.225495   
             cellina (graph)    0.753276  0.138765  0.855499  0.140828   
             cellina            0.741235  0.156217  0.833842  0.157907   

                               precision           direction_match_k  \
                                    mean       std              mean   
perturbation model_name                                                
CRC          mean shift         0.249167  0.096187          0.208500   
             CPA                0.249000  0.139187          0.238667   
             scGen              0.295600  0.133762          0.265800   
             SpatialProp        0.138167  0.083310          0.122833   
             MintFlow           0.267000  0.153731          0.247600   
             cellina (ablated)  0.334333  0.132299          0.322833   
             cellina (graph)    0.448833  0.152342          0.442833   
             cellina            0.389000  0.133819          0.384333   

                                         edistance_local            
                                     std            mean       std  
perturbation model_name                                             
CRC          mean shift         0.087879       25.370604  5.901509  
             CPA                0.144896        4.314905  3.611717  
             scGen              0.152470        3.578312  2.931190  
             SpatialProp        0.087972       16.660148  4.241646  
             MintFlow           0.162340       23.967456  4.851615  
             cellina (ablated)  0.140010        4.348332  3.732404  
             cellina (graph)    0.159302        4.532351  3.633426  
             cellina            0.139844        4.604068  3.617880

In [10]:
def find_best(agg: pd.DataFrame) -> dict:
    best = {}
    if isinstance(agg.index, pd.MultiIndex):
        # agg index: (perturbation, model_name)
        # collect perturbations preserving order
        raw = [p for p, _ in agg.index.tolist()]
        seen = set()
        perturbations = [p for p in raw if not (p in seen or seen.add(p))]
        for metric, direction in METRICS.items():
            for pert in perturbations:
                try:
                    means = agg.loc[pert][(metric, "mean")].dropna()
                except KeyError:
                    continue
                if means.empty:
                    continue
                best[(pert, metric)] = means.idxmax() if direction > 0 else means.idxmin()
    else:
        for metric, direction in METRICS.items():
            means = agg[(metric, "mean")].dropna()
            if means.empty:
                continue
            best[metric] = means.idxmax() if direction > 0 else means.idxmin()
    return best


def format_cell(mean, std, bold, na_str="--"):
    if pd.isna(mean):
        return na_str
    s = f"{mean:.3f} $\\pm$ {std:.3f}" if not pd.isna(std) else f"{mean:.3f}"
    return f"\\textbf{{{s}}}" if bold else s

In [11]:
best = find_best(agg)
table = pd.DataFrame(index=agg.index, columns=[PRETTY_METRIC[m] for m in METRICS])

for metric in METRICS:
    col = PRETTY_METRIC[metric]
    for model in agg.index:
        pert, model_name = model

        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]

        bold = (best.get((pert, metric)) == model_name)

        table.loc[model, col] = format_cell(mean, std, bold)
table

Spearman $\uparrow$  \
perturbation model_name                                      
CRC          mean shift                  0.450 $\pm$ 0.268   
             CPA                         0.544 $\pm$ 0.196   
             scGen                       0.497 $\pm$ 0.259   
             SpatialProp                 0.415 $\pm$ 0.253   
             MintFlow                    0.582 $\pm$ 0.270   
             cellina (ablated)           0.647 $\pm$ 0.212   
             cellina (graph)    \textbf{0.753 $\pm$ 0.139}   
             cellina                     0.741 $\pm$ 0.156   

                                        Pearson $\uparrow$  \
perturbation model_name                                      
CRC          mean shift                  0.435 $\pm$ 0.270   
             CPA                         0.630 $\pm$ 0.218   
             scGen                       0.506 $\pm$ 0.363   
             SpatialProp                 0.473 $\pm$ 0.285   
             MintFlow                    0.660 $\pm$ 0.283   
             cellina (ablated)           0.740 $\pm$ 0.225   
             cellina (graph)    \textbf{0.855 $\pm$ 0.141}   
             cellina                     0.834 $\pm$ 0.158   

                                      Precision $\uparrow$  \
perturbation model_name                                      
CRC          mean shift                  0.249 $\pm$ 0.096   
             CPA                         0.249 $\pm$ 0.139   
             scGen                       0.296 $\pm$ 0.134   
             SpatialProp                 0.138 $\pm$ 0.083   
             MintFlow                    0.267 $\pm$ 0.154   
             cellina (ablated)           0.334 $\pm$ 0.132   
             cellina (graph)    \textbf{0.449 $\pm$ 0.152}   
             cellina                     0.389 $\pm$ 0.134   

                               Direction Match (K) $\uparrow$  \
perturbation model_name                                         
CRC          mean shift                     0.208 $\pm$ 0.088   
             CPA                            0.239 $\pm$ 0.145   
             scGen                          0.266 $\pm$ 0.152   
             SpatialProp                    0.123 $\pm$ 0.088   
             MintFlow                       0.248 $\pm$ 0.162   
             cellina (ablated)              0.323 $\pm$ 0.140   
             cellina (graph)       \textbf{0.443 $\pm$ 0.159}   
             cellina                        0.384 $\pm$ 0.140   

                               E-dist (local) $\downarrow$  
perturbation model_name                                     
CRC          mean shift                 25.371 $\pm$ 5.902  
             CPA                         4.315 $\pm$ 3.612  
             scGen              \textbf{3.578 $\pm$ 2.931}  
             SpatialProp                16.660 $\pm$ 4.242  
             MintFlow                   23.967 $\pm$ 4.852  
             cellina (ablated)           4.348 $\pm$ 3.732  
             cellina (graph)             4.532 $\pm$ 3.633  
             cellina                     4.604 $\pm$ 3.618

In [12]:
def insert_midrule_perts(latex, table):
    groups = table.index.get_level_values(0)
    boundaries = [i for i in range(1, len(groups)) if groups[i] != groups[i - 1]]

    lines = latex.splitlines()
    new_lines = []

    row_idx = 0

    for line in lines:
        new_lines.append(line)

        # detect data rows (skip header)
        if "&" in line and "\\\\" in line:
            if row_idx in boundaries:
                new_lines.append(r"\midrule")
            row_idx += 1

    latex = "\n".join(new_lines)
    return latex

In [13]:
is_multi_pert = table.index.get_level_values(0).nunique() > 1

if is_multi_pert:
    latex = table.to_latex(
        index_names=False, # removes the extra header row
        escape=False,
        header=False,
        column_format="ll" + "c" * table.shape[1],  # two index levels → 'll'
        multirow=True,  # for multiple perturbations
        caption=(
            f"Leave-one-celltype-out performance (top {N_DEG} DEGs). For each slide we "
            "first average over the held-out cell types, then report mean $\\pm$ std "
            f"across {df['sid'].nunique()} slides. Best per metric in \\textbf{{bold}}."
        ),
        label=f"tab:loo_summary_{DATASET_NAME}",
    )

    latex = latex.replace(r"\cline{1-7}", "")
    latex = latex.replace("\\begin{table}", "\\begin{table}[t]\n\\centering")
    latex = latex.replace(r"\multirow[t]", r"\multirow[c]")
    header = "Holdout \\\ perturbation & Method & " + " & ".join(table.columns) + r" \\"
    latex = latex.replace(
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}",
    "\\begin{tabular}{ll" + "c" * table.shape[1] + "}\n\\toprule\n" + header
    )
    latex = latex.replace(r"\midrule", "")
    latex = insert_midrule_perts(latex, table)
else:
    flat = table.reset_index(level=0, drop=True)

    latex = flat.to_latex(
        index=True,
        index_names=False,
        escape=False,
        column_format="l" + "c" * flat.shape[1],
        caption=(
            f"Leave-one-celltype-out performance (top {N_DEG} DEGs). "
            "Mean $\\pm$ std across slides. Best per metric in \\textbf{bold}."
        ),
        label=f"tab:loo_summary_{DATASET_NAME}",
    )

print(latex)

(OUT_DIR / f"loo_summary_{DATASET_NAME}_DEG_{N_DEG}_table.tex").write_text(latex)

\begin{table}
\caption{Leave-one-celltype-out performance (top 200 DEGs). Mean $\pm$ std across slides. Best per metric in \textbf{bold}.}
\label{tab:loo_summary_crc}
\begin{tabular}{lccccc}
\toprule
 & Spearman $\uparrow$ & Pearson $\uparrow$ & Precision $\uparrow$ & Direction Match (K) $\uparrow$ & E-dist (local) $\downarrow$ \\
\midrule
mean shift & 0.450 $\pm$ 0.268 & 0.435 $\pm$ 0.270 & 0.249 $\pm$ 0.096 & 0.208 $\pm$ 0.088 & 25.371 $\pm$ 5.902 \\
CPA & 0.544 $\pm$ 0.196 & 0.630 $\pm$ 0.218 & 0.249 $\pm$ 0.139 & 0.239 $\pm$ 0.145 & 4.315 $\pm$ 3.612 \\
scGen & 0.497 $\pm$ 0.259 & 0.506 $\pm$ 0.363 & 0.296 $\pm$ 0.134 & 0.266 $\pm$ 0.152 & \textbf{3.578 $\pm$ 2.931} \\
SpatialProp & 0.415 $\pm$ 0.253 & 0.473 $\pm$ 0.285 & 0.138 $\pm$ 0.083 & 0.123 $\pm$ 0.088 & 16.660 $\pm$ 4.242 \\
MintFlow & 0.582 $\pm$ 0.270 & 0.660 $\pm$ 0.283 & 0.267 $\pm$ 0.154 & 0.248 $\pm$ 0.162 & 23.967 $\pm$ 4.852 \\
cellina (ablated) & 0.647 $\pm$ 0.212 & 0.740 $\pm$ 0.225 & 0.334 $\pm$ 0.132 & 0.323 $\p

1336

In [14]:
# Markdown preview (arrows rendered as unicode so they look right in the notebook).
md_metric = {
    "spearman": "Spearman ↑",
    "pearson": "Pearson ↑",
    "precision": "Precision ↑",
    "direction_match_k": "Direction Match (K) ↑",
    "edistance_local": "E-dist (local) ↓",
    #"rmse": "RMSE ↓",
}

md_table = pd.DataFrame(index=agg.index, columns=[md_metric[m] for m in METRICS])
for metric in METRICS:
    col = md_metric[metric]
    for model in agg.index:
        mean = agg.loc[model, (metric, "mean")]
        std = agg.loc[model, (metric, "std")]
        if pd.isna(mean):
            md_table.loc[model, col] = "--"
            continue
        s = f"{mean:.3f} ± {std:.3f}"
        md_table.loc[model, col] = f"**{s}**" if best.get(metric) == model else s

md_table.index.name = "Method"
print(md_table.to_markdown())

| Method                       | Spearman ↑    | Pearson ↑     | Precision ↑   | Direction Match (K) ↑   | E-dist (local) ↓   |
|:-----------------------------|:--------------|:--------------|:--------------|:------------------------|:-------------------|
| ('CRC', 'mean shift')        | 0.450 ± 0.268 | 0.435 ± 0.270 | 0.249 ± 0.096 | 0.208 ± 0.088           | 25.371 ± 5.902     |
| ('CRC', 'CPA')               | 0.544 ± 0.196 | 0.630 ± 0.218 | 0.249 ± 0.139 | 0.239 ± 0.145           | 4.315 ± 3.612      |
| ('CRC', 'scGen')             | 0.497 ± 0.259 | 0.506 ± 0.363 | 0.296 ± 0.134 | 0.266 ± 0.152           | 3.578 ± 2.931      |
| ('CRC', 'SpatialProp')       | 0.415 ± 0.253 | 0.473 ± 0.285 | 0.138 ± 0.083 | 0.123 ± 0.088           | 16.660 ± 4.242     |
| ('CRC', 'MintFlow')          | 0.582 ± 0.270 | 0.660 ± 0.283 | 0.267 ± 0.154 | 0.248 ± 0.162           | 23.967 ± 4.852     |
| ('CRC', 'cellina (ablated)') | 0.647 ± 0.212 | 0.740 ± 0.225 | 0.334 ± 0.132 | 0.323 ± 0.140          